In [ ]:
import pickle
import geoopt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, TensorDataset
from sklearn.metrics import f1_score, mean_squared_error, r2_score
from scipy.stats import spearmanr
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix
import os
import matplotlib.pyplot as plt
from tqdm import tqdm
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap
from sklearn.metrics import ndcg_score
import math
import copy
from geoopt import ManifoldParameter, PoincareBall


In [ ]:
# --- CONFIGURATION ---
class Config:
    # UPDATE THIS PATH TO YOUR FILE
    data_path = 'finalized_pooled_embeddings.pkl'

    epochs_trans = 500
    epochs_cis = 100
    batch_size = 512

    # --- BINS INFO ---
    interval = [-np.inf, -2, -.5, .5, 2, np.inf]
    inter_names = ["Destroied", 'Deleterious', 'Neutral', 'Beneficial', "Improved"]
    label_col = ["#290000","#B01818", "#0AEE0A", '#00008B', "#096BA8"]
    
    # Hyperbolic Params
    hyp_lr = 1e-4
    hyp_dim = 1024
    n_head = 8
    c = 1.

    # Euclidean Params
    euc_lr = 1e-4
    euc_hidden = [2048, 512, 128, 32]

    output_dim = len(inter_names)
    val_frac = 0.1
    seed = 80085
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
def evaluate_classification_metrics(model, loader, device, model_type="euclidean"):
    model.eval()
    all_preds = []
    all_true = []

    with torch.no_grad():
        for feats, labels in loader:
            feats = feats.to(device)
            labels = labels.to(device)

            if model_type == "euclidean":
                logits = model(feats.float())
            else:
                logits = model(feats.double())
            
            # Apply Argmax to get class predictions (0, 1, 2)
            preds = torch.argmax(logits, dim=1)

            all_preds.append(preds.cpu().numpy())
            all_true.append(labels.cpu().numpy())

    y_pred = np.concatenate(all_preds)
    y_true = np.concatenate(all_true)

    # Calculate both Weighted and Macro F1 scores
    f1_weighted = f1_score(y_true, y_pred, average='weighted', zero_division=0)
    f1_macro = f1_score(y_true, y_pred, average='macro', zero_division=0)
    
    report = classification_report(y_true, y_pred, zero_division=0, output_dict=True)

    return f1_weighted, f1_macro, report

def discretize_fitness(fitness_array):
    """Transforms continuous fitness into 4 discrete classes for F1 comparison."""
    bins = Config.interval
    labels = np.digitize(fitness_array, bins) - 1
    return labels

class FeatureDataset(Dataset):
    def __init__(self, pkl_path):
        print(f"Loading {pkl_path}...")
        with open(pkl_path, 'rb') as f:
            data = pickle.load(f)

        # Handle different potential formats (Dict or DataFrame)
        if hasattr(data, 'loc'): # Pandas DataFrame
            self.fitness = data['fitness'].values
            embeddings_raw = data['embedding'].values
        else: # Dictionary
            self.fitness = np.array(data['fitness'])
            embeddings_raw = np.array(data['embedding'])

        # 1. Prepare Features
        # Assume embeddings_raw is already a list/array of fixed-size vectors
        # Stack them into a single tensor (N_samples, Dim)
        # Verify shape is (N, D), not (N, L, D)
        self.embeddings = np.stack(embeddings_raw).astype(np.float64) 
        
        # 2. Prepare Labels (Classification)
        self.labels = discretize_fitness(self.fitness)

        print(f"Dataset Loaded. Shape: {self.embeddings.shape}")
        
    def __len__(self):
        return len(self.embeddings)

    def __getitem__(self, idx):
        # Return Tuple: (Feature, Label)
        return (
            torch.tensor(self.embeddings[idx], dtype=torch.float64), 
            torch.tensor(self.labels[idx], dtype=torch.long)
        )

In [ ]:
# --- 1. THE FIXED HOROSPHERICAL LAYER (Now supports Curvature) ---
class HorosphericalLayer(nn.Module):
    def __init__(self, hyp_dim, num_classes, c=1.0):
        super().__init__()
        self.hyp_dim = hyp_dim
        self.num_classes = num_classes
        self.c = c
        
        # Manifold for Prototypes (The Boundary)
        self.sphere = geoopt.Sphere()

        angles = torch.linspace(0, 2*torch.pi, num_classes+1)[:-1]
        circle_points = torch.stack([
            torch.cos(angles),
            torch.sin(angles)
        ], dim=1)
        padding = torch.zeros(num_classes, hyp_dim - 2)
        initial_protos = torch.cat([circle_points, padding], dim=1)
        self.prototypes = geoopt.ManifoldParameter(
            initial_protos,
            manifold=self.sphere,
            requires_grad=False
        )
        # Learnable Biases (a)
        self.biases = nn.Parameter(torch.zeros(num_classes))

    def busemann_function(self, x, p):
        """
        Numerically Stable Generalized Busemann function.
        Uses Log-Split trick: log(num/den) = log(num) - log(den)
        Removes the hard clamp on x_norm to satisfy your friend.
        """
        sqrt_c = self.c ** 0.5
        
        # 1. No Hard Clamp on x_norm (Friend's request)
        # We only clamp x to be *barely* inside the boundary if it technically drifted out.
        # This is a safety net for float errors, not a geometric wall.
        # Boundary is 1/sqrt(c). We allow it to get extremely close (1e-15 buffer).
        max_r = (1.0 / sqrt_c) - 1e-15
        
        # We use "renorm" instead of clamp to preserve direction
        x_norm = torch.norm(x, p=2, dim=-1, keepdim=True)
        cond = x_norm > max_r
        # Project back only if we strictly exceeded the safe radius
        x = torch.where(cond, x * (max_r / x_norm), x)
        
        # Re-calculate safe norm squared
        x_norm_sq = torch.sum(x ** 2, dim=-1, keepdim=True)
        
        # 2. Compute Numerator & Denominator
        dots = torch.matmul(x, p.t()) 
        
        # Numerator: 1 + c||x||^2 - 2*sqrt(c)*<p,x>
        numerator = 1.0 + self.c * x_norm_sq - 2 * sqrt_c * dots
        
        # Denominator: 1 - c||x||^2
        denominator = 1.0 - self.c * x_norm_sq
        
        # 3. The Magic: Clamp VALUES, not coordinates
        # We allow denominator to be tiny, but not zero/negative.
        numerator = numerator.clamp_min(1e-15)
        denominator = denominator.clamp_min(1e-15)
        
        # 4. Split Logarithm
        # log(num / den) = log(num) - log(den)
        # This is much more stable than dividing two tiny numbers first.
        term1 = torch.log(numerator)
        term2 = torch.log(denominator)
        
        dist = (1.0 / sqrt_c) * (term1 - term2)
        
        return dist

    def forward(self, x):
        # Ensure input is double precision
        x = x.double()
        
        # 1. Calculate Attraction (Busemann Distance)
        dists = self.busemann_function(x, self.prototypes)
        
        # 2. Compute Logits: -Distance + Bias
        logits = -dists + self.biases
        
        return logits

In [ ]:

# --- TRUE HYPERBOLIC MODEL ---

class HyperbolicMLR(nn.Module):
    """
    True Hyperbolic Classification using Möbius Linear layers.
    It projects to the Poincaré ball and performs classification 
    directly on the manifold using distance-based logits.
    """
    def __init__(self, input_dim, hidden_dims, hyp_dim, num_classes, c):
        super().__init__()
        self.c = c # Curvature
        self.num_classes = num_classes
        self.hyp_dim = hyp_dim
        self.ball = PoincareBall(c=self.c)
        # 1. Euclidean Encoder (Feature Extractor)
        # We process inputs in Euclidean space first to get rich features
        layers = []
        in_d = input_dim
        for h_dim in hidden_dims:
            layers.append(nn.Linear(in_d, h_dim))
            layers.append(nn.LayerNorm(h_dim))
            layers.append(nn.SiLU())
            layers.append(nn.Dropout(0.05))
            in_d = h_dim
        self.encoder = nn.Sequential(*layers)
        
        # 2. Projection to Hyperbolic Space
        # porta i dati da R(256) a R(hyperbolic dim)
        self.to_tangent = nn.Linear(in_d, hyp_dim)

        self.horo_layer = HorosphericalLayer(hyp_dim, num_classes, c=c)

        self.double()

    # --- Hyperbolic Operations ---
    def exp_map_zero(self, v):
        """Map from tangent space at origin to hyperbolic space"""
        v = v.double()
        v_norm = torch.norm(v, p=2, dim=-1, keepdim=True).clamp_min(1e-6)
        sqrt_c = self.c ** 0.5 
        return torch.tanh(sqrt_c * v_norm) * (v / (sqrt_c * v_norm))

    def forward(self, x):
        x = x.double() 
        
        # 1. Encode in Euclidean
        feat = self.encoder(x)
        
        # 2. Map to Tangent Space (at origin)
        tangent = self.to_tangent(feat)
        
        # 3. Project to Hyperbolic Manifold (Poincaré Ball)
        hyp_emb = self.exp_map_zero(tangent)
        
        # FARE LAYER HYPERBOLICI 

        # 4. Classification on the Manifold
        # The logits are calculated using hyperbolic geometry logic
        logits = self.horo_layer(hyp_emb)
        
        return logits # Shape: (Batch_Size, 3)

# --- STANDARD EUCLIDEAN CLASSIFIER ---

class EuclideanClassifier(nn.Module):
    def __init__(self, input_dim, hidden_dims, num_classes):
        super().__init__()
        layers = []
        in_d = input_dim
        for h_dim in hidden_dims:
            layers.append(nn.Linear(in_d, h_dim))
            layers.append(nn.LayerNorm(h_dim))
            layers.append(nn.SiLU())
            layers.append(nn.Dropout(0.05))
            in_d = h_dim
        
        # Final layer outputs scores for 3 classes
        layers.append(nn.Linear(in_d, num_classes))
        self.model = nn.Sequential(*layers)

    def forward(self, x):
        x = x.float()
        return self.model(x) # Shape: (Batch_Size, 3)


def evaluate_proteingym_metrics(model, loader, device, model_type):
    model.eval()
    all_preds = []
    all_true = []

    with torch.no_grad():
        # Updated to unpack just feats and labels (as per your previous fix)
        for feats, labels in loader:
            feats = feats.to(device)
            labels = labels.to(device)

            if model_type == "euclidean":
                logits = model(feats.float())
            else:
                logits = model(feats.double())

            # --- ADAPTATION START ---
            # Convert Logits (3 classes) -> Continuous Score for Ranking/MSE
            # We calculate the "Expected Class Value": 0*P(Del) + 1*P(Neu) + 2*P(Ben)
            # This results in a float between 0.0 and 2.0
            probs = torch.softmax(logits, dim=1)
            classes = torch.arange(logits.shape[1], device=device)
            # Example: Probs [0.1, 0.2, 0.7] -> Score = 0*0.1 + 1*0.2 + 2*0.7 = 1.6
            preds = torch.sum(probs * classes, dim=1)
            # --- ADAPTATION END ---

            all_preds.append(preds.cpu().numpy())
            all_true.append(labels.cpu().numpy())

    y_pred_raw = np.concatenate(all_preds)
    y_true_raw = np.concatenate(all_true)

    # 1. Spearman's Correlation
    # Measures if the model ranks Class 2 samples higher than Class 0 samples
    rho, _ = spearmanr(y_true_raw, y_pred_raw)

    # 2. MSE
    # Measures error between True Label (0, 1, or 2) and Predicted Score (e.g., 1.6)
    mse = mean_squared_error(y_true_raw, y_pred_raw)

    # 3. NDCG
    # Use True Labels (0,1,2) as relevance scores
    # Safety check: if y_true is constant (e.g. all samples are class 1), ndcg crashes or is undefined
    if len(np.unique(y_true_raw)) < 2:
         ndcg = 0.0
    else:
        # We ensure relevance is non-negative (it is already 0,1,2 but good practice)
        y_true_shifted = y_true_raw - y_true_raw.min()
        ndcg = ndcg_score(
            np.expand_dims(y_true_shifted, axis=0), 
            np.expand_dims(y_pred_raw, axis=0)
        )

    return rho, mse, ndcg

In [ ]:
def discretize_fitness(fitness_array):
    """Transforms continuous fitness into 4 discrete classes for F1 comparison."""
    bins = Config.interval
    labels = np.digitize(fitness_array, bins) - 1
    return labels


def visualize_confusion_matrix(model, loader, device, model_type, title="Confusion Matrix"):
    model.eval()
    all_preds = []
    all_true = []

    # 1. Raccogli tutte le predizioni
    with torch.no_grad():
        # Updated to unpack just feats and labels
        for feats, labels in loader:
            feats = feats.to(device)
            labels = labels.to(device)

            if model_type == "euclidean":
                logits = model(feats.float())
            else:
                logits = model(feats.double())

            # CHANGE: Get class index directly via argmax
            preds = torch.argmax(logits, dim=1)

            all_preds.append(preds.cpu().numpy())
            all_true.append(labels.cpu().numpy())

    y_pred_cls = np.concatenate(all_preds)
    y_true_cls = np.concatenate(all_true)

    # 3. Calcola la Matrice
    cm = confusion_matrix(y_true_cls, y_pred_cls, labels= range(Config.output_dim))

    # 4. Plotting con Seaborn
    plt.figure(figsize=(8, 6))

    # Definiamo etichette leggibili in base ai tuoi bin
    # Bins: [-inf, -3, -1, 1, 3, inf] -> 5 Classi
    class_names = Config.inter_names

    # Assicuriamoci che le etichette corrispondano alle classi presenti
    # (Se nel validation set mancano alcune classi, questo evita errori)
    unique_classes = sorted(list(set(y_true_cls) | set(y_pred_cls)))
    labels = [class_names[i] for i in unique_classes if i < len(class_names)]

    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=labels, yticklabels=labels)

    plt.xlabel('Predicted Label')
    plt.ylabel('True Label')
    plt.title(f'{title} (Model: {model_type})')
    plt.show()

def visualize_fitness_distribution_colored(model, loader, device, model_type):
    model.eval()
    all_preds = []
    all_true = []

    # 1. Inference
    with torch.no_grad():
        for feats, labels in loader:
            feats = feats.to(device)
            labels = labels.to(device)

            if model_type == "euclidean":
                logits = model(feats.float())
            else:
                logits = model(feats.double())

            # Get predicted class index
            preds = torch.argmax(logits, dim=1)

            all_preds.append(preds.cpu().numpy())
            all_true.append(labels.cpu().numpy())

    pred_classes = np.concatenate(all_preds)
    true_classes = np.concatenate(all_true)

    # --- DYNAMIC CONFIGURATION ---
    # Determine N classes based on Config, or fallback to unique values in data
    if hasattr(Config, 'inter_names') and Config.inter_names:
        class_labels = Config.inter_names
        n_classes = len(class_labels)
    else:
        n_classes = len(np.unique(true_classes))
        class_labels = [f"Class {i}" for i in range(n_classes)]

    # Handle Colors: If we have more classes than colors, extend the palette
    base_colors = Config.label_col
    if len(base_colors) < n_classes:
        # Fallback to a matplotlib colormap if we run out of manual colors
        import matplotlib.cm as cm
        cmap_obj = cm.get_cmap('viridis', n_classes)
        colors = [cmap_obj(i) for i in range(n_classes)]
    else:
        colors = base_colors[:n_classes]
    
    cmap = ListedColormap(colors)
    # -----------------------------

    # 2. Sort data by TRUE Class
    data = sorted(zip(true_classes, pred_classes), key=lambda x: (x[0], np.random.rand()))
    y_true_sorted, pred_cls_sorted = zip(*data)

    # 3. Jitter Setup
    n_samples = len(y_true_sorted)
    x_base = np.arange(n_samples)

    # Jitter X (Horizontal)
    jitter_strength = n_samples * 0.02
    noise_x = np.random.normal(0, jitter_strength, size=n_samples)
    x_axis_dispersed = x_base + noise_x

    # Jitter Y (Vertical) - Spreads points within their class band
    # We use 0.1 as a standard "width" for the band
    noise_y = np.random.normal(0, 0.1, size=n_samples) 
    y_axis_dispersed = np.array(y_true_sorted) + noise_y

    plt.figure(figsize=(12, 1 + n_classes * 1.5)) # Scale height by N classes

    # 4. Scatter Plot
    scatter = plt.scatter(x_axis_dispersed, y_axis_dispersed, c=pred_cls_sorted, cmap=cmap,
                          vmin=0, vmax=n_classes-1, alpha=0.6, s=25, edgecolors='none')

    # 5. Legend & Labels
    patches = [mpatches.Patch(color=colors[i], label=class_labels[i]) for i in range(n_classes)]
    plt.legend(handles=patches, title="Predicted Class", loc='upper left')

    plt.xlabel('Sample Index (Sorted by True Class)')
    plt.ylabel('True Class')
    
    # Dynamic Y-Ticks
    plt.yticks(range(n_classes), class_labels) 
    
    plt.title(f'Dispersed Class Distribution ({model_type})')
    plt.grid(True, alpha=0.15, axis='x')

    # Dynamic Limits: Frame from -0.5 to N-0.5
    plt.xlim(-jitter_strength*3, n_samples + jitter_strength*3)
    plt.ylim(-0.5, n_classes - 0.5) 

    plt.show()

def compute_class_weights(loader, device):
    print("Computing class weights for imbalance correction...")
    # Initialize counters for 3 classes
    counts = torch.zeros(len(Config.inter_names)).to(device)
    
    # Iterate over the loader to count labels
    # (Since we are using the simple TensorDataset loader now, this is fast)
    for _, labels in loader:
        labels = labels.to(device)
        # bincount works on 1D integer tensors
        counts += torch.bincount(labels, minlength=len(Config.inter_names))
        
    print(f"Class Counts: {counts.cpu().numpy()}")
    
    # 1. Compute Inverse Frequency
    # Add epsilon to avoid division by zero if a class is missing (unlikely)
    weights = 1.0 / (counts + 1e-6)
    
    # 2. Normalize weights
    # We normalize so that the sum of weights equals the number of classes (3).
    # This keeps the learning rate scale consistent.
    weights = weights / weights.sum() * len(Config.inter_names)
    
    print(f"Computed Class Weights: {weights.cpu().numpy()}")
    
    # Weights must be Float (Float32 or Float64 depending on model)
    return weights

In [ ]:
def train_hyperbolic(config, train_feat_loader, val_feat_loader, feat_dim, C, class_weights=None):
    print("\n--- Training Hyperbolic Classifier (RiemannianAdam) ---")
    torch.set_default_dtype(torch.float64)

    # Initialize the Geoopt Model
    model = HyperbolicMLR(feat_dim, [512], config.hyp_dim, num_classes=Config.output_dim, c = C).to(config.device)
    
    # --- OPTIMIZER CHANGE ---
    # Use RiemannianAdam. It automatically detects ManifoldParameters (like self.weight)
    # and standard Parameters (like self.bias or encoder weights) and updates them appropriately.
    optimizer = geoopt.optim.RiemannianAdam(model.parameters(), lr=config.hyp_lr)
    
    # Loss Function
    if class_weights is not None:
        class_weights = class_weights.to(config.device).double()
        criterion = nn.CrossEntropyLoss(weight=class_weights)
    else:
        criterion = nn.CrossEntropyLoss()

    best_f1_w = -1.0
    best_f1_m = -1.0
    best_rho = -1.0
    best_model_state = None
    best_report = None

    for epoch in range(config.epochs_trans):
        model.train()
        total_loss = 0.0

        for feats, labels in train_feat_loader:
            feats = feats.to(config.device).double()
            labels = labels.to(config.device).long()

            # 1. Forward Pass
            logits = model(feats)
            loss = criterion(logits, labels)

            # 2. Backward Pass
            optimizer.zero_grad()
            loss.backward()
            
            # Optional: Gradient Clipping (works for Euclidean grads, safe for RiemannianAdam too)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            
            # 3. Update Step (RiemannianAdam handles the exponential maps here)
            optimizer.step()
            
            total_loss += loss.item()

        # Evaluate Metrics
        val_f1_w, val_f1_m, _ = evaluate_classification_metrics(model, val_feat_loader, config.device, "hyperbolic")
        val_rho, val_mse, _ = evaluate_proteingym_metrics(model, val_feat_loader, config.device, "hyperbolic")

        # Logging
        if (epoch + 1) % 10 == 0:
            print(f"Epoch {epoch+1} | Loss: {total_loss / len(train_feat_loader):.4f} | "
                  f"Val F1(W): {val_f1_w:.4f} | Val F1(M): {val_f1_m:.4f} | "
                  f"Val Spearman: {val_rho:.4f}")

        # Save Best Model Logic
        if val_f1_w > best_f1_w:
            best_f1_w = val_f1_w
        
        if val_f1_m > best_f1_m:
            best_f1_m = val_f1_m

        if val_rho > best_rho:
            best_rho = val_rho
            best_model_state = copy.deepcopy(model.state_dict())

    # Load best weights
    if best_model_state:
        model.load_state_dict(best_model_state)

    return best_rho, best_f1_w, best_f1_m, best_report, model

def train_euclidean(config, train_feat_loader, val_feat_loader, feat_dim, class_weights=None):
    print("\n--- Training Euclidean Classifier ---")
    torch.set_default_dtype(torch.float32)

    model = EuclideanClassifier(feat_dim, config.euc_hidden, num_classes = Config.output_dim).to(config.device)
    optimizer = torch.optim.Adam(model.parameters(), lr=config.euc_lr)
    
    if class_weights is not None:
        class_weights = class_weights.to(config.device).float()
        criterion = nn.CrossEntropyLoss(weight=class_weights)
    else:
        criterion = nn.CrossEntropyLoss()

    best_f1_w = -1.0
    best_f1_m = -1.0
    best_rho = -1.0
    best_model_state = None
    best_report = None

    for epoch in range(config.epochs_cis):
        model.train()
        total_loss = 0.0

        for feats, labels in train_feat_loader:
            feats = feats.to(config.device).float()
            labels = labels.to(config.device).long()

            logits = model(feats)
            loss = criterion(logits, labels)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        # Evaluate: Unpack both F1 scores
        val_f1_w, val_f1_m, _ = evaluate_classification_metrics(model, val_feat_loader, config.device, "euclidean")
        val_rho, val_mse, _ = evaluate_proteingym_metrics(model, val_feat_loader, config.device, "euclidean")

        if (epoch + 1) % 5 == 0:
            print(f"Epoch {epoch+1} | Loss: {total_loss / len(train_feat_loader):.4f} | "
                  f"Val F1(W): {val_f1_w:.4f} | Val F1(M): {val_f1_m:.4f} | "
                  f"Val Spearman: {val_rho:.4f} | Val MSE: {val_mse:.4f}")

        if val_rho > best_rho:
            best_rho = val_rho
            best_model_state = copy.deepcopy(model.state_dict())

        if val_f1_w > best_f1_w:
            best_f1_w = val_f1_w
            
        if val_f1_m > best_f1_m:
            best_f1_m = val_f1_m

    if best_model_state:
        model.load_state_dict(best_model_state)

    return best_rho, best_f1_w, best_f1_m, best_report, model

In [ ]:
from sklearn.decomposition import PCA
import matplotlib.lines as mlines

def visualize_poincare_embeddings(model, loader, device, title="Poincaré Embeddings"):
    model.eval()
    
    # --- 1. Extract Embeddings & Prototypes ---
    all_embs = []
    all_labels = []
    
    with torch.no_grad():
        for feats, labels in loader:
            feats = feats.to(device).double()
            # Forward pass to get embeddings (before classification head)
            encoded = model.encoder(feats)
            tangent = model.to_tangent(encoded)
            hyp_embs = model.exp_map_zero(tangent)
            
            all_embs.append(hyp_embs.cpu())
            all_labels.append(labels.cpu())
            
        prototypes = model.horo_layer.prototypes.data.cpu()

    X = torch.cat(all_embs, dim=0).numpy()
    y = torch.cat(all_labels, dim=0).numpy()
    P = prototypes.numpy()

    print(f"Projecting {X.shape[0]} points from {X.shape[1]}D to 2D disk...")

    # --- 2. Dimensionality Reduction (Log -> PCA -> Exp) ---
    def log_map_zero(x, c=1.0):
        x_norm = np.linalg.norm(x, axis=-1, keepdims=True)
        x_norm = np.clip(x_norm, 1e-9, 1.0/np.sqrt(c) - 1e-9)
        scale = np.arctanh(np.sqrt(c) * x_norm) / (np.sqrt(c) * x_norm)
        return x * scale

    def exp_map_zero(v, c=1.0):
        v_norm = np.linalg.norm(v, axis=-1, keepdims=True)
        v_norm = np.clip(v_norm, 1e-9, None)
        scale = np.tanh(np.sqrt(c) * v_norm) / (np.sqrt(c) * v_norm)
        return v * scale

    X_tangent = log_map_zero(X, c=model.c)
    P_tangent = log_map_zero(P, c=model.c)
    
    pca = PCA(n_components=2)
    X_pca = pca.fit_transform(X_tangent) 
    P_pca = pca.transform(P_tangent)
    
    X_disk = exp_map_zero(X_pca, c=model.c)
    P_disk = exp_map_zero(P_pca, c=model.c)

    # --- 3. Plotting with Legend ---
    plt.figure(figsize=(10, 10))
    ax = plt.gca()
    
    # Draw Boundary
    boundary_radius = 1.0 / (model.c ** 0.5)
    circle = plt.Circle((0, 0), boundary_radius, color='black', fill=False, linewidth=2, alpha=0.3)
    ax.add_artist(circle)
    
    # Get Config Info
    unique_classes = np.unique(y)
    class_names = Config.inter_names if hasattr(Config, 'inter_names') else [str(c) for c in unique_classes]
    colors = Config.label_col if hasattr(Config, 'label_col') else ['red', 'blue', 'green', 'orange']
    
    # Loop through classes to ensure correct legend
    for i, cls_idx in enumerate(unique_classes):
        # Filter points for this class
        mask = (y == cls_idx)
        class_color = colors[i] if i < len(colors) else 'gray'
        class_label = class_names[i] if i < len(class_names) else f"Class {cls_idx}"
        
        # Plot Data Points (Dots)
        plt.scatter(X_disk[mask, 0], X_disk[mask, 1], 
                    c=class_color, 
                    label=class_label, # This adds it to the legend
                    alpha=0.6, s=20, edgecolors='none')

        # Plot Prototype (Star) - Same color, bigger size
        # We don't give it a label to avoid duplicate entries in legend
        if i < len(P_disk):
            plt.scatter(P_disk[i, 0], P_disk[i, 1], 
                        c=class_color, 
                        s=400, marker='*', edgecolors='black', linewidth=1.5, zorder=10)

    # Clean up plot
    plt.xlim(-1.1, 1.1)
    plt.ylim(-1.1, 1.1)
    plt.gca().set_aspect('equal', adjustable='box')
    plt.title(title)
    plt.grid(False)
    plt.axis('off')

    # --- Legend Customization ---
    # Create the legend and place it outside or in a corner
    plt.legend(title="Protein Fitness", loc="upper right", frameon=True)
    
    plt.show()

In [ ]:
def main_func():
    # --- CONFIGURATION ---
    # Ensure this matches your file path
    pkl_path = Config.data_path 
    
    # 1. Load Data
    full_dataset = FeatureDataset(pkl_path)
    
    # 2. Split Train/Val
    # We use a simple random split here
    val_size = int(len(full_dataset) * Config.val_frac)
    train_size = len(full_dataset) - val_size
    
    train_dataset, val_dataset = torch.utils.data.random_split(
        full_dataset, [train_size, val_size], 
        generator=torch.Generator().manual_seed(Config.seed)
    )
    
    print(f"Train samples: {len(train_dataset)}, Val samples: {len(val_dataset)}")

    # 3. Create Loaders
    train_loader = DataLoader(train_dataset, batch_size=Config.batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=Config.batch_size, shuffle=False)

    # 4. Compute Class Weights (To handle imbalance)
    class_weights = compute_class_weights(train_loader, Config.device)

    # 5. Get Input Dimension automatically
    # (Grab one sample to check size)
    sample_feat, _ = train_dataset[0]
    feat_dim = sample_feat.shape[0] 
    print(f"Detected Feature Dimension: {feat_dim}")

    # ==========================================
    # MODEL 1: HYPERBOLIC CLASSIFIER
    # ==========================================
    best_rho_hyp, best_f1_w_hyp, best_f1_m_hyp, report_hyp, hyp_model = train_hyperbolic(
        Config, 
        train_loader, 
        val_loader, 
        feat_dim=feat_dim, 
        class_weights=class_weights,
        C = Config.c
    )

    # ==========================================
    # MODEL 2: EUCLIDEAN CLASSIFIER
    # ==========================================
    best_rho_euc, best_f1_w_euc, best_f1_m_euc, report_euc, euc_model = train_euclidean(
        Config, 
        train_loader, 
        val_loader, 
        feat_dim=feat_dim, 
        class_weights=class_weights
    )

    print("\n--- Hyperbolic Results ---")
    visualize_confusion_matrix(hyp_model, val_loader, Config.device, "hyperbolic")
    visualize_fitness_distribution_colored(hyp_model, val_loader, Config.device, "hyperbolic")

    print("\n--- Euclidean Results ---")
    visualize_confusion_matrix(euc_model, val_loader, Config.device, "euclidean")
    visualize_fitness_distribution_colored(euc_model, val_loader, Config.device, "euclidean")

    # ==========================================
    # FINAL COMPARISON
    # ==========================================
    print("\n" + "="*40)
    print(f"FINAL METRICS COMPARISON:")
    print(f"{'Metric':<15} | {'Hyperbolic':<10} | {'Euclidean':<10}")
    print("-" * 40)
    print(f"{'F1 (Weighted)':<15} | {best_f1_w_hyp:.4f}     | {best_f1_w_euc:.4f}")
    print(f"{'F1 (Macro)':<15} | {best_f1_m_hyp:.4f}     | {best_f1_m_euc:.4f}")
    print(f"{'Spearman Rho':<15} | {best_rho_hyp:.4f}     | {best_rho_euc:.4f}")
    print("="*40)
    print("\n--- Visualizing Hyperbolic Embeddings ---")
    visualize_poincare_embeddings(hyp_model, val_loader, Config.device, title="Poincaré Embeddings (Validation)")

if __name__ == "__main__":
    main_func()